# 🎬 VP Library — Embed ภาพลง Supabase

Notebook นี้ embed ภาพทุกฉากด้วย **CLIP ViT-B/32** โหลดโมเดลใน Colab โดยตรง ไม่ต้องพึ่ง HF API

### วิธีใช้
1. ใส่ `SUPABASE_SERVICE_KEY` ใน Cell 2
2. กด **Runtime → Run all**
3. รอ ~5–10 นาที (โหลดโมเดลครั้งแรก ~2 นาที)

> ไม่ต้องมี HF Token

In [ ]:
# ── Cell 1: ติดตั้ง package ──────────────────────────────────────
!pip install transformers torch pillow requests supabase -q
print('✅ ติดตั้งเสร็จ')

In [ ]:
# ── Cell 2: ใส่ Supabase Service Key ────────────────────────────
# หาได้ที่: Supabase Dashboard → Settings → API Keys → Secret keys (กดตา 👁 เพื่อดู)

SUPABASE_URL         = 'https://pgaqdqbjyewwckpslyvx.supabase.co'
SUPABASE_SERVICE_KEY = 'YOUR_SERVICE_ROLE_KEY_HERE'

SKIP_ALREADY_INDEXED = True  # True = ข้าม scene ที่ embed แล้ว

if 'YOUR_SERVICE' in SUPABASE_SERVICE_KEY:
    print('⚠️  ยังไม่ได้ใส่ SUPABASE_SERVICE_KEY!')
else:
    print('✅ Config พร้อม')

In [ ]:
# ── Cell 3: โหลด CLIP model + เชื่อมต่อ Supabase ──────────────────
import torch
import requests
from PIL import Image
from io import BytesIO
from datetime import datetime, timezone
from transformers import CLIPProcessor, CLIPModel
from supabase import create_client

# โหลด CLIP ViT-B/32 (image encoder เดียวกับ clip-ViT-B-32-multilingual-v1)
print('🔄 โหลด CLIP model...')
CLIP_MODEL_NAME = 'openai/clip-vit-base-patch32'
processor  = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
clip_model.eval()
print('✅ โหลด model เสร็จ')

# เชื่อมต่อ Supabase
sb = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)
print('✅ Supabase เชื่อมต่อแล้ว')

def encode_image(image_url: str) -> list:
    """ดาวน์โหลดภาพ → encode ด้วย CLIP → คืน vector[512] normalized"""
    img_bytes = requests.get(image_url, timeout=30).content
    img       = Image.open(BytesIO(img_bytes)).convert('RGB')
    inputs    = processor(images=img, return_tensors='pt')
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
        features = features / features.norm(p=2, dim=-1, keepdim=True)  # L2 normalize
    return features[0].tolist()

# ทดสอบด้วยภาพตัวอย่าง
test_url = 'https://pgaqdqbjyewwckpslyvx.supabase.co/storage/v1/object/public/vp-images/scenes/VP-001-020/VP-001-full.webp'
try:
    v = encode_image(test_url)
    print(f'✅ ทดสอบ encode สำเร็จ — dim={len(v)}, norm≈{sum(x**2 for x in v)**0.5:.3f}')
except Exception as e:
    print(f'⚠️  ทดสอบ encode ล้มเหลว: {e} (อาจ URL ภาพตัวอย่างผิด — ไม่กระทบ Cell ถัดไป)')

In [ ]:
# ── Cell 4: ดึงรายการฉากจาก Supabase ──────────────────────────────
result   = sb.from_('scenes').select('id, image_url, mclip_indexed_at').order('sort_order').execute()
all_sc   = result.data or []
to_embed = [
    s for s in all_sc
    if s.get('image_url') and (not SKIP_ALREADY_INDEXED or not s.get('mclip_indexed_at'))
]

print(f'📊 ฉากทั้งหมด:   {len(all_sc)}')
print(f'✅ embed แล้ว:  {len(all_sc) - len(to_embed)}')
print(f'⏳ รอ embed:    {len(to_embed)}')
if not to_embed:
    print('\n🎉 ทุกฉาก embed แล้ว!')
else:
    print('\nฉากที่จะ embed:')
    for s in to_embed:
        print(f'  • {s["id"]}')

In [ ]:
# ── Cell 5: Embed ทุกฉาก → บันทึกลง Supabase ───────────────────────
if not to_embed:
    print('🎉 ไม่มีฉากที่ต้อง embed')
else:
    success, failed, errors = 0, 0, []

    for i, scene in enumerate(to_embed):
        sid = scene['id']
        print(f'[{i+1}/{len(to_embed)}] 🔄 {sid}...', end=' ', flush=True)
        try:
            vector = encode_image(scene['image_url'])
            now    = datetime.now(timezone.utc).isoformat()
            sb.from_('scenes').update({
                'mclip_vector':     vector,
                'mclip_indexed_at': now,
            }).eq('id', sid).execute()
            success += 1
            print(f'✅ dim={len(vector)}')
        except Exception as e:
            failed += 1
            errors.append((sid, str(e)))
            print(f'❌ {e}')

    print(f'\n══════════════════════')
    print(f'✅ สำเร็จ:  {success} ฉาก')
    print(f'❌ ล้มเหลว: {failed} ฉาก')
    if errors:
        print('\nรายละเอียด error:')
        for sid, err in errors:
            print(f'  {sid}: {err}')
    if success > 0:
        print('\n🎉 เสร็จ! ตอนนี้ vp-search.html ค้นหาได้แล้ว')

In [ ]:
# ── Cell 6 (Optional): ตรวจสอบผลลัพธ์ ─────────────────────────────
rows      = (sb.from_('scenes').select('id, mclip_indexed_at').order('sort_order').execute()).data or []
indexed   = [r for r in rows if r.get('mclip_indexed_at')]
unindexed = [r for r in rows if not r.get('mclip_indexed_at')]
print(f'✅ มี vector: {len(indexed)} ฉาก')
print(f'⏳ ยังไม่มี:  {len(unindexed)} ฉาก')
if unindexed:
    for r in unindexed: print(f'   • {r["id"]}')
else:
    print('\n🎉 ทุกฉากครบแล้ว!')